In [ ]:
# =============================================================================
# split first -> augment ONLY the training images -> class folders
#
#
# Output (all in OUT_DIR):
#   splits.csv              the split (re-used if it already exists)
#   dataset_by_class/       ALL images (originals + augmented):
#                               <split>/<shape>/score_<n>/image.jpg
#   splits_augmented.csv    originals + augmented rows
#                           (extra columns: is_augmented, source_path, class_path)
# =============================================================================
import re
import shutil
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

# ------------------------------------------------------------------ settings
DATA_DIR = Path("/kaggle/input/datasets/huss69/shape-dataset/Shapes")
OUT_DIR = Path("/kaggle/working/results")
MAX_MULT = 10          # a rare score grows to at most MAX_MULT x its original size
FLAT = False           # False: <split>/<shape>/score_<n>/    True: <shape>/score_<n>/
RESPLIT = False        # True: ignore an existing splits.csv and split again
SEED = 42
SPLIT_RATIOS = (0.70, 0.15, 0.15)

MAX_ROTATION = 7.0             # degrees  (orientation facet)
SCALE_RANGE = (0.93, 1.07)     # uniform zoom (size + edges facets)
SHIFT = 0.03                   # fraction of width / height
FREE_ROTATION = {"circle"}     # circle has no orientation facet
HFLIP_SHAPES = {"circle"}      # add "square","triangle","diamond","star" ONLY after checking the
                               # stimuli are left-right symmetric; never add wave / pencils

# ------------------------------------------------------------------ shapes / max scores
FOLDERS = {
    "circle": ("circle", 4), "square": ("square", 5), "triangle": ("triangle", 5),
    "diagonal": ("diamond", 5), "diamond": ("diamond", 5), "wave": ("wave", 4), "star": ("star", 5),
    "overlapped circle": ("overlapped_circles", 6), "overlapped circles": ("overlapped_circles", 6),
    "overlapped pencil": ("overlapped_pencils", 6), "overlapped pencils": ("overlapped_pencils", 6),
}
CODES = {"C": ("circle", 4), "S": ("square", 5), "SQ": ("square", 5), "T": ("triangle", 5),
         "D": ("diamond", 5), "W": ("wave", 4), "ST": ("star", 5), "OC": ("overlapped_circles", 6),
         "OP": ("overlapped_pencils", 6), "P": ("overlapped_pencils", 6)}
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
NAME_RE = re.compile(r"^img[_\- ]?(\d+)[_\- ]+([A-Za-z]+)[_\- ]+(\d+)", re.IGNORECASE)
LAST_NUM_RE = re.compile(r"(\d+)\D*$")
ID_RE = re.compile(r"(\d+)")


# ------------------------------------------------------------------ step 1: scan + split
def parse_filename(p: Path):
    m = NAME_RE.match(p.stem)
    if m:
        i, code, score = m.groups()
        return int(i), code.upper(), int(score)
    last = LAST_NUM_RE.search(p.stem)
    if not last:
        return None
    first = ID_RE.search(p.stem)
    return int(first.group(1)) if first else -1, None, int(last.group(1))


def scan_dataset(data_dir: Path) -> pd.DataFrame:
    rows = []
    for p in sorted(data_dir.rglob("*")):
        if p.suffix.lower() not in IMG_EXTS or p.name.startswith("._"):
            continue
        parsed = parse_filename(p)
        if parsed is None:
            continue
        img_id, code, score = parsed
        rel = p.relative_to(data_dir).parts
        info = FOLDERS.get(rel[0].strip().lower()) if len(rel) > 1 else None
        if info is None and code is not None:
            info = CODES.get(code)
        if info is None:
            info = ((rel[0] if len(rel) > 1 else (code or "unknown")).strip().lower().replace(" ", "_"), None)
        rows.append(dict(path=str(p.resolve()), image_id=img_id, folder=rel[0] if len(rel) > 1 else "",
                         shape_code=code or "", shape=info[0], score=score, cfg_max=info[1]))
    df = pd.DataFrame(rows)
    if df.empty:
        raise SystemExit(f"No labelled images found in {data_dir}")
    bad = df[df.cfg_max.notna() & (df.score > df.cfg_max)]          # impossible scores (e.g. star = 6)
    for _, r in bad.iterrows():
        print(f"[warn] excluded {Path(r.path).name}: {r['shape']} score {r.score} > max {int(r.cfg_max)}")
    df = df.drop(bad.index)
    mx = {s: int(g.cfg_max.dropna().max()) if g.cfg_max.notna().any() else int(g.score.max())
          for s, g in df.groupby("shape")}
    df["max_score"] = df["shape"].map(mx)
    return df.drop(columns="cfg_max").reset_index(drop=True)


def stratified_split(df, ratios=SPLIT_RATIOS, seed=SEED) -> pd.Series:
    """Per (shape, score): n=1 -> train; n=2 -> 1 train + 1 test; n>=3 -> >=1 in val and in test."""
    rng = np.random.default_rng(seed)
    split = pd.Series("train", index=df.index)
    for _, g in df.groupby(["shape", "score"]):
        idx = rng.permutation(g.index.values)
        n = len(idx)
        if n == 1:
            continue
        if n == 2:
            split[idx[1]] = "test"
            continue
        n_test = max(1, int(round(n * ratios[2])))
        n_val = max(1, int(round(n * ratios[1])))
        split[idx[:n_test]] = "test"
        split[idx[n_test:n_test + n_val]] = "val"
    return split


# ------------------------------------------------------------------ helpers
def class_dir(root: Path, split: str, shape: str, score, flat: bool) -> Path:
    sub = Path(shape) / f"score_{int(score)}"
    return root / sub if flat else root / split / sub


def ink_mask(img):
    """Boolean mask of the red-pencil strokes."""
    dark = 255 - img.min(axis=2)
    k = max(15, (min(dark.shape) // 15) | 1)
    bg = cv2.morphologyEx(dark, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k)))
    ink = cv2.subtract(dark, bg)
    if ink.max() < 10:
        return np.zeros(ink.shape, bool)
    return cv2.threshold(ink, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1] > 0


def touches_border(mask, frac=0.01):
    g = max(2, int(frac * min(mask.shape)))
    return bool(mask[:g].any() or mask[-g:].any() or mask[:, :g].any() or mask[:, -g:].any())


def augment_image(img, shape, rng):
    """One facet-safe augmented copy."""
    h, w = img.shape[:2]
    angle = rng.uniform(-180, 180) if shape in FREE_ROTATION else rng.uniform(-MAX_ROTATION, MAX_ROTATION)
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, rng.uniform(*SCALE_RANGE))   # uniform scale
    M[:, 2] += rng.uniform(-SHIFT, SHIFT, 2) * (w, h)
    out = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
    if shape in HFLIP_SHAPES and rng.random() < 0.5:
        out = np.ascontiguousarray(out[:, ::-1])
    out = out.astype(np.float32) * rng.uniform(0.85, 1.15) + rng.uniform(-15, 15)   # contrast, brightness
    out = 255.0 * np.clip(out / 255.0, 0, 1) ** rng.uniform(0.9, 1.1)               # gamma
    out += rng.normal(0, 3.0, out.shape)                                            # light noise
    return np.clip(out, 0, 255).astype(np.uint8)


# ------------------------------------------------------------------ run
OUT_DIR.mkdir(parents=True, exist_ok=True)
ROOT = OUT_DIR / "dataset_by_class"
splits_csv = OUT_DIR / "splits.csv"

# 1) SPLIT FIRST
if splits_csv.exists() and not RESPLIT:
    df = pd.read_csv(splits_csv)
    print(f"[split] re-using existing {splits_csv}")
else:
    df = scan_dataset(DATA_DIR)
    df["split"] = stratified_split(df)
    df.to_csv(splits_csv, index=False)
    print(f"[split] created {splits_csv}")
df = df.reset_index(drop=True)
df["is_augmented"] = False
df["source_path"] = df["path"]

# originals -> class folders
new_paths = []
for r in df.itertuples():
    d = class_dir(ROOT, r.split, r.shape, r.score, FLAT)
    d.mkdir(parents=True, exist_ok=True)
    dst = d / Path(r.path).name
    if not dst.exists():
        shutil.copy2(r.path, dst)
    new_paths.append(str(dst))
df["class_path"] = new_paths

# 2) THEN AUGMENT THE TRAINING IMAGES ONLY
rng = np.random.default_rng(SEED)
aug_rows, rejected = [], 0
train = df[df.split == "train"]
for shape, sdf in train.groupby("shape"):
    counts = sdf.score.value_counts()
    target = counts.max()
    for score, c in counts.items():
        need = int(min(target - c, c * (MAX_MULT - 1)))
        if need <= 0:
            continue                                   # majority score: no copies
        src = sdf[sdf.score == score].reset_index(drop=True)
        out_dir = class_dir(ROOT, "train", shape, score, FLAT)
        out_dir.mkdir(parents=True, exist_ok=True)
        border_flag = {}
        for k in range(need):
            i = k % len(src)
            r = src.iloc[i]
            img = cv2.imread(r.path, cv2.IMREAD_COLOR)
            if img is None:
                continue
            if i not in border_flag:
                border_flag[i] = touches_border(ink_mask(img))
            aug = None
            for _ in range(20):                        # reject copies that cut the drawing
                cand = augment_image(img, shape, rng)
                if border_flag[i] or not touches_border(ink_mask(cand)):
                    aug = cand
                    break
            if aug is None:
                rejected += 1
                continue
            dst = out_dir / f"{Path(r.path).stem}_aug{int(score)}_{k:03d}.jpg"
            cv2.imwrite(str(dst), aug, [cv2.IMWRITE_JPEG_QUALITY, 95])
            row = r.to_dict()
            row.update(path=str(dst), class_path=str(dst), split="train", is_augmented=True, source_path=r.path)
            aug_rows.append(row)
if rejected:
    print(f"[augment] {rejected} copies skipped (drawing would have been cut by the image border)")

aug_df = pd.DataFrame(aug_rows)
full = pd.concat([df, aug_df], ignore_index=True) if len(aug_df) else df
full.to_csv(OUT_DIR / "splits_augmented.csv", index=False)

# 3) SAFETY CHECKS + SUMMARY
assert not full[full.split != "train"].is_augmented.any(), "augmented image found outside train!"
if len(aug_df):
    assert aug_df.source_path.isin(df[df.split == "train"].path).all(), "augmented from a non-train image!"

tr_before = df[df.split == "train"]
tr_after = full[full.split == "train"]
print("\nTrain images per shape / score BEFORE augmentation:\n", pd.crosstab(tr_before["shape"], tr_before.score))
print("\nTrain images per shape / score AFTER augmentation:\n", pd.crosstab(tr_after["shape"], tr_after.score))
print("\nVal / test images (unchanged):\n", pd.crosstab(df["shape"], df["split"])[["val", "test"]])
print(f"\nAll {len(full)} images ({len(df)} original + {len(aug_df)} augmented) are in: {ROOT}")
print("Folder layout:", "<shape>/score_<n>/" if FLAT else "<split>/<shape>/score_<n>/")
print(f"Saved {OUT_DIR / 'splits_augmented.csv'}")

[warn] excluded img1688-ST-6.png: star score 6 > max 5
[split] created /kaggle/working/results/splits.csv
[augment] 16 copies skipped (drawing would have been cut by the image border)

Train images per shape / score BEFORE augmentation:
 score               0  2   3    4    5    6
shape                                      
circle              3  1  78  319    0    0
diamond             2  0   1   50  334    0
overlapped_circles  1  1   0    7   95  270
overlapped_pencils  8  1   6   38  101  158
square              1  0   0   31  376    0
star                9  6  72  176   71    0
triangle            1  0   0   26  377    0
wave                1  2  78  304    0    0

Train images per shape / score AFTER augmentation:
 score                0   2    3    4    5    6
shape                                         
circle              23   3  317  319    0    0
diamond             20   0   10  334  334    0
overlapped_circles  10  10    0   70  270  270
overlapped_pencils  80  10   60  1